# Download Data

## 1. Config

In [1]:
import sys
from pathlib import Path
import pandas as pd
from datetime import datetime
from random import random
from time import time
import shutil

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

import os
import tarfile
import requests
from tqdm import tqdm
from dotenv import load_dotenv
from src.download.downloader import (
    get_missing_data_list,
    download_data_archive,
    extract_data_archive
)

In [ ]:
load_dotenv()

BASE_URL = os.getenv("DAIC_WOZ_URL")
EXTENDED_BASE_URL = os.getenv("EXTENDED_DAIC_WOZ_URL")
DATA_START_ID = 300
DATA_END_ID = 492

RAW_DIR = PROJECT_ROOT / "data" / "raw"
AUDIO_DIR = PROJECT_ROOT / "data" / "audio"
TRANSCRIPT_DIR = PROJECT_ROOT / "data" / "transcript"

if RAW_DIR.exists():
    shutil.rmtree(RAW_DIR)

RAW_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
TRANSCRIPT_DIR.mkdir(parents=True, exist_ok=True)


LOG_DIR = PROJECT_ROOT / "logs"
SUCCESS_LOG = LOG_DIR / "download_success.csv"
FAILED_LOG = LOG_DIR / "download_failures.csv"


assert BASE_URL is not None, "DAIC_WOZ_URL not found in .env"
assert EXTENDED_BASE_URL is not None, "EXTENDED_DAIC_WOZ_URL not found in .env"

## 2. Download Audio and Transcripts

In [3]:
miss_data_list = get_missing_data_list(
    start_id=DATA_START_ID,
    end_id=DATA_END_ID,
    audio_dir=AUDIO_DIR,
    transcript_dir=TRANSCRIPT_DIR
)
print(f"Missing data IDs: {miss_data_list}")

Completed : 0
Corrupted : 0
Missing   : 2
Need Download : 2
Missing data IDs: [500, 501]


In [4]:
success_count = 0
failed_count = 0

for data_id in tqdm(
    miss_data_list,
    desc="Downloading DAIC-WOZ",
    unit="data",
):
    archive_path = None

    try:
        download_result = download_data_archive(
            data_id=data_id,
            base_url=EXTENDED_BASE_URL,
            output_dir=RAW_DIR,
        )

        archive_path = download_result["file_path"]

        extract_result = extract_data_archive(
            archive_path=archive_path,
            audio_dir=AUDIO_DIR,
            transcript_dir=TRANSCRIPT_DIR,
        )

        if archive_path.exists():
            archive_path.unlink()

        success_row = pd.DataFrame([
            {
                "data_id": data_id,
                "audio_path": str(extract_result["audio_path"]),
                "transcript_path": str(extract_result["transcript_path"]),
                "timestamp": datetime.now().isoformat(),
            }
        ])

        success_row.to_csv(
            SUCCESS_LOG,
            mode="a",
            header=not SUCCESS_LOG.exists(),
            index=False,
        )

        success_count += 1
        tqdm.write(f"[SUCCESS] {data_id}")

    except Exception as e:
        if archive_path is not None and archive_path.exists():
            archive_path.unlink()

        tmp_path = RAW_DIR / f"{data_id}_P.tar.gz.tmp"
        if tmp_path.exists():
            tmp_path.unlink()

        failed_row = pd.DataFrame([
            {
                "data_id": data_id,
                "error_type": type(e).__name__,
                "error": str(e),
                "timestamp": datetime.now().isoformat(),
            }
        ])

        failed_row.to_csv(
            FAILED_LOG,
            mode="a",
            header=not FAILED_LOG.exists(),
            index=False,
        )

        failed_count += 1
        tqdm.write(f"[FAILED] {data_id} | {type(e).__name__}: {e}")
    

print("\n====================")
print(f"Success : {success_count}")
print(f"Failed  : {failed_count}")
print("====================")

[DOWNLOAD] 500_P.tar.gz


[FAILED] 500 | HTTPError: 404 Client Error: Not Found for url: https://dcapswoz.ict.usc.edu/wwwedaic/data/500_P.tar.gz
[DOWNLOAD] 501_P.tar.gz


[FAILED] 501 | HTTPError: 404 Client Error: Not Found for url: https://dcapswoz.ict.usc.edu/wwwedaic/data/501_P.tar.gz

Success : 0
Failed  : 2
